In [16]:
import cv2
import easyocr
import re
from ultralytics import YOLO

# ==========================================
# 1. MULTI-PASS PREPROCESSING
# ==========================================
def prep_raw(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    return cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)

def prep_clahe(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    return clahe.apply(resized)

def prep_thresh(crop):
    gray = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    resized = cv2.resize(gray, None, fx=2.0, fy=2.0, interpolation=cv2.INTER_CUBIC)
    thresh = cv2.adaptiveThreshold(resized, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 19, 9)
    return cv2.bitwise_not(thresh)

# ==========================================
# 2. STRICT REGEX HEURISTICS
# ==========================================
def apply_heuristics(text):
    clean_text = text.replace(" ", "").upper()
    clean_text = re.sub(r"[^A-Z0-9]", "", clean_text)
    
    # Target standard NY plates (3 letters, 4 numbers)
    ny_pattern = re.search(r'[A-Z]{3}[0-9]{4}', clean_text)
    
    if ny_pattern:
        return ny_pattern.group(0)
    return None

# ==========================================
# 3. MAIN EXECUTION LOOP
# ==========================================
def main():
    print("Loading Final Hybrid ALPR Engine...")
    yolo_model = YOLO(r'C:/Users/Tyler/Documents/Schoolwork/IST 691/Final_Main/runs/detect/IST691_Paper/baseline_nano-2/weights/best.pt')
    reader = easyocr.Reader(['en'], gpu=True)

    images = ['TylerCar.jpg', 'KimiCar.jpg']
    results = yolo_model.predict(source=images, conf=0.4, verbose=False)

    for i, result in enumerate(results):
        print(f"\n==============================")
        print(f"ALPR RESULTS FOR: {images[i]}")
        print(f"==============================\n")
        
        original_image = result.orig_img
        
        for box in result.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0].tolist())
            raw_crop = original_image[y1:y2, x1:x2]
            
            passes = [
                ("Pass 1 (Raw)", prep_raw(raw_crop)),
                ("Pass 2 (CLAHE)", prep_clahe(raw_crop)),
                ("Pass 3 (Thresh)", prep_thresh(raw_crop))
            ]
            
            best_plate = None
            max_text_height = 0
            
            # Image center for gating (wider gate: 0.9)
            img_width = x2 - x1
            image_center_x = img_width / 2
            
            for pass_name, processed_img in passes:
                ocr_results = reader.readtext(processed_img, allowlist='ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789 -')
                
                for res in ocr_results:
                    bbox = res[0]
                    raw_text = res[1]
                    
                    word_center_x = (bbox[0][0] + bbox[1][0]) / 2
                    text_height = bbox[2][1] - bbox[0][1]
                    
                    valid_plate = apply_heuristics(raw_text)
                    
                    # THE RELAXED CENTERING GATE (0.9 vs 0.6)
                    is_centered = abs(word_center_x - image_center_x) < (image_center_x * 0.9)
                    
                    if valid_plate and is_centered and text_height > max_text_height:
                        max_text_height = text_height
                        best_plate = valid_plate
                        
            if best_plate:
                print(f"► FINAL WINNING TEXT: {best_plate}")
            else:
                print("► No valid pattern found in center of plate.")

if __name__ == '__main__':
    main()

Loading Final Hybrid ALPR Engine...

ALPR RESULTS FOR: TylerCar.jpg

► No valid pattern found in center of plate.

ALPR RESULTS FOR: KimiCar.jpg

► No valid pattern found in center of plate.
